# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')


In [ ]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
        Document(page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
        Document(page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')
    ]
retrieve_vectordb()

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')]

In [3]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', temperature = 0)
prompt = ChatPromptTemplate.from_template('''
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '서울의 인구, 관광지, 교통인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'

retrieved_docs = retrieve_vectordb(question)
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question': question})

print(response)

### 1. 핵심 데이터 정리
- **인구**: 서울 인구는 약 1,000만 명입니다.
- **관광지**: 경복궁, 남산타워, 명동 등 다양한 관광지가 있습니다.
- **교통·문화·인프라**: 교통과 문화 인프라가 잘 갖추어져 있습니다.
- **도시 환경**: 한강을 중심으로 도시가 발달했습니다.

### 2. 상호관계 분석
서울은 인구가 많은 대도시인 만큼 다양한 관광·문화 시설과 편의시설이 발달해 있습니다. 또한 교통 인프라가 잘 갖추어져 있어 경복궁, 남산타워, 명동처럼 서로 다른 성격의 관광지를 편리하게 이동하며 둘러볼 수 있습니다. 여기에 한강과 같은 자연환경까지 더해져 역사, 쇼핑, 전망, 휴식 등 다양한 여행 목적을 한 도시에서 충족할 수 있습니다.

### 3. 논리적 서술
서울은 약 1,000만 명이 거주하는 대도시로, 여행객을 위한 교통과 문화 인프라가 잘 마련되어 있습니다. 따라서 주요 관광지 간 이동이 편리하고, 여행 중 필요한 편의시설을 이용하기도 쉽습니다. 또한 경복궁에서는 역사와 전통문화를, 남산타워에서는 서울의 도시 경관을, 명동에서는 쇼핑과 도심 문화를 경험할 수 있습니다. 한강은 도심 속에서 여유와 휴식을 즐길 수 있는 공간을 제공합니다.

### 4. 최종 답변
서울은 **다양한 관광지와 편리한 교통·문화 인프라가 조화를 이루고 있어 여행하기 좋은 도시**입니다. 경복궁, 남산타워, 명동 등은 각각 역사, 전망, 쇼핑이라는 서로 다른 여행 경험을 제공하며, 잘 갖추어진 교통망 덕분에 이들을 효율적으로 방문할 수 있습니다. 또한 약 1,000만 명이 거주하는 대도시인 만큼 다양한 편의시설과 문화시설을 이용하기 쉽고, 한강을 통해 도심 속 휴식도 즐길 수 있습니다. 따라서 서울은 짧은 일정에도 여러 종류의 관광을 편리하게 경험할 수 있다는 점에서 여행하기 좋은 곳이라고 할 수 있습니다.
